# 02_build_splits

This notebook creates the fixed pilot split for the compression study.

## Pilot split
- Train: `video01`-`video10`
- Validation: `video11`-`video15`
- Test: `video16`-`video20`

## Goals
- Define the split once, cleanly
- Save split files to disk
- Save both video-level and frame-level/sample-level split metadata
- Verify no overlap between train / val / test

## Notes
This notebook does **not** extract frames yet.
It only defines and saves the split structure that later notebooks will load.

In [1]:
# ============================================================
# 1. Imports
# ============================================================
import json
from pathlib import Path

import pandas as pd

In [2]:
# ============================================================
# 2. Paths
# ============================================================
PROJECT_ROOT = Path.cwd()
DATA_ROOT = PROJECT_ROOT / "cholec80"

VIDEOS_DIR = DATA_ROOT / "videos"
PHASE_ANN_DIR = DATA_ROOT / "phase_annotations"
SPLITS_DIR = DATA_ROOT / "splits"
OUTPUTS_DIR = DATA_ROOT / "outputs"

SPLITS_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUTS_DIR / "split_audit").mkdir(parents=True, exist_ok=True)

In [3]:
# ============================================================
# 3. Fixed pilot split
# ============================================================
TRAIN_VIDEOS = [f"video{str(i).zfill(2)}" for i in range(1, 11)]
VAL_VIDEOS   = [f"video{str(i).zfill(2)}" for i in range(11, 16)]
TEST_VIDEOS  = [f"video{str(i).zfill(2)}" for i in range(16, 21)]

print("Train:", TRAIN_VIDEOS)
print("Val:  ", VAL_VIDEOS)
print("Test: ", TEST_VIDEOS)

Train: ['video01', 'video02', 'video03', 'video04', 'video05', 'video06', 'video07', 'video08', 'video09', 'video10']
Val:   ['video11', 'video12', 'video13', 'video14', 'video15']
Test:  ['video16', 'video17', 'video18', 'video19', 'video20']


In [4]:
# ============================================================
# 4. Verify all videos exist in the dataset
# ============================================================
available_videos = sorted([p.stem for p in VIDEOS_DIR.glob("video*.mp4")])

split_check = {
    "train_missing": [v for v in TRAIN_VIDEOS if v not in available_videos],
    "val_missing": [v for v in VAL_VIDEOS if v not in available_videos],
    "test_missing": [v for v in TEST_VIDEOS if v not in available_videos],
}

split_check

{'train_missing': [], 'val_missing': [], 'test_missing': []}

In [5]:
# ============================================================
# 5. Verify no overlap
# ============================================================
train_set = set(TRAIN_VIDEOS)
val_set = set(VAL_VIDEOS)
test_set = set(TEST_VIDEOS)

print("Train ∩ Val :", train_set & val_set)
print("Train ∩ Test:", train_set & test_set)
print("Val ∩ Test  :", val_set & test_set)

Train ∩ Val : set()
Train ∩ Test: set()
Val ∩ Test  : set()


## Save video-level split files

In [6]:
# ============================================================
# 6. Save simple video-level split CSVs
# ============================================================
train_videos_df = pd.DataFrame({"video": TRAIN_VIDEOS, "split": "train"})
val_videos_df   = pd.DataFrame({"video": VAL_VIDEOS, "split": "val"})
test_videos_df  = pd.DataFrame({"video": TEST_VIDEOS, "split": "test"})

train_videos_df.to_csv(SPLITS_DIR / "pilot_train_videos.csv", index=False)
val_videos_df.to_csv(SPLITS_DIR / "pilot_val_videos.csv", index=False)
test_videos_df.to_csv(SPLITS_DIR / "pilot_test_videos.csv", index=False)

print("Saved:")
print(SPLITS_DIR / "pilot_train_videos.csv")
print(SPLITS_DIR / "pilot_val_videos.csv")
print(SPLITS_DIR / "pilot_test_videos.csv")

Saved:
/Users/niranjani/Desktop/video-compression-project/cholec80/splits/pilot_train_videos.csv
/Users/niranjani/Desktop/video-compression-project/cholec80/splits/pilot_val_videos.csv
/Users/niranjani/Desktop/video-compression-project/cholec80/splits/pilot_test_videos.csv


In [7]:
# ============================================================
# 7. Save one combined split CSV
# ============================================================
split_videos_df = pd.concat(
    [train_videos_df, val_videos_df, test_videos_df],
    ignore_index=True
).sort_values(["split", "video"]).reset_index(drop=True)

split_videos_df.to_csv(SPLITS_DIR / "pilot_split_videos.csv", index=False)
split_videos_df

,video,split
0,video16,test
1,video17,test
2,video18,test
3,video19,test
4,video20,test
5,video01,train
6,video02,train
7,video03,train
8,video04,train
9,video05,train


In [8]:
# ============================================================
# 8. Save split JSON
# ============================================================
split_json = {
    "train": TRAIN_VIDEOS,
    "val": VAL_VIDEOS,
    "test": TEST_VIDEOS,
}

with open(SPLITS_DIR / "pilot_split.json", "w") as f:
    json.dump(split_json, f, indent=2)

print("Saved:", SPLITS_DIR / "pilot_split.json")

Saved: /Users/niranjani/Desktop/video-compression-project/cholec80/splits/pilot_split.json


## Build frame-level annotation table for the pilot videos

Even though we are not extracting sampled frames yet, we should save a clean annotation table for the selected 20 videos.

Because the audit showed that:
- annotations are available at full frame rate
- `frame_idx` increases by 1
- one annotation row corresponds to one raw frame

we will later downsample to 1 fps using:
- `frame_idx % 25 == 0`

In [9]:
# ============================================================
# 9. Phase mapping
# ============================================================
PHASE_MAP = {
    "Preparation": 0,
    "CalotTriangleDissection": 1,
    "ClippingCutting": 2,
    "GallbladderDissection": 3,
    "GallbladderPackaging": 4,
    "CleaningCoagulation": 5,
    "GallbladderRetraction": 6,
}
ID_TO_PHASE = {v: k for k, v in PHASE_MAP.items()}

In [10]:
# ============================================================
# 10. Parse phase file
# ============================================================
def parse_phase_file(phase_file: Path) -> pd.DataFrame:
    rows = []
    with open(phase_file, "r") as f:
        lines = f.readlines()

    if lines and ("Frame" in lines[0] or "Phase" in lines[0]):
        lines = lines[1:]

    for line in lines:
        parts = line.strip().split()
        if len(parts) < 2:
            continue

        frame_idx = int(parts[0])
        phase_name = parts[1].strip()

        if phase_name not in PHASE_MAP:
            continue

        rows.append({
            "frame_idx": frame_idx,
            "phase_name": phase_name,
            "phase_id": PHASE_MAP[phase_name],
        })

    return pd.DataFrame(rows)

In [11]:
# ============================================================
# 11. Build annotation table for pilot split
# ============================================================
pilot_videos = TRAIN_VIDEOS + VAL_VIDEOS + TEST_VIDEOS

annotation_tables = []

for video in pilot_videos:
    phase_file = PHASE_ANN_DIR / f"{video}-phase.txt"
    df = parse_phase_file(phase_file)
    df["video"] = video

    if video in TRAIN_VIDEOS:
        df["split"] = "train"
    elif video in VAL_VIDEOS:
        df["split"] = "val"
    else:
        df["split"] = "test"

    annotation_tables.append(df)

pilot_ann_df = pd.concat(annotation_tables, ignore_index=True)

pilot_ann_df = pilot_ann_df[
    ["video", "split", "frame_idx", "phase_name", "phase_id"]
].reset_index(drop=True)

print("Pilot annotation rows:", len(pilot_ann_df))
pilot_ann_df.head()

Pilot annotation rows: 1151996


,video,split,frame_idx,phase_name,phase_id
0,video01,train,0,Preparation,0
1,video01,train,1,Preparation,0
2,video01,train,2,Preparation,0
3,video01,train,3,Preparation,0
4,video01,train,4,Preparation,0


In [12]:
# ============================================================
# 12. Save full-frame pilot annotation table
# ============================================================
pilot_ann_df.to_csv(SPLITS_DIR / "pilot_annotations_fullfps.csv", index=False)
print("Saved:", SPLITS_DIR / "pilot_annotations_fullfps.csv")

Saved: /Users/niranjani/Desktop/video-compression-project/cholec80/splits/pilot_annotations_fullfps.csv


## Build 1 fps pilot annotation table

We now define the exact rows that later notebooks should use for 1 fps experiments.

Rule:
- keep rows where `frame_idx % 25 == 0`

In [13]:
# ============================================================
# 13. Downsample pilot annotations to 1 fps
# ============================================================
pilot_ann_1fps_df = pilot_ann_df[pilot_ann_df["frame_idx"] % 25 == 0].copy()

pilot_ann_1fps_df["second_idx"] = (pilot_ann_1fps_df["frame_idx"] // 25).astype(int)

pilot_ann_1fps_df = pilot_ann_1fps_df[
    ["video", "split", "frame_idx", "second_idx", "phase_name", "phase_id"]
].reset_index(drop=True)

print("Pilot 1 fps annotation rows:", len(pilot_ann_1fps_df))
pilot_ann_1fps_df.head(20)

Pilot 1 fps annotation rows: 46099


,video,split,frame_idx,second_idx,phase_name,phase_id
0,video01,train,0,0,Preparation,0
1,video01,train,25,1,Preparation,0
2,video01,train,50,2,Preparation,0
3,video01,train,75,3,Preparation,0
4,video01,train,100,4,Preparation,0
5,video01,train,125,5,Preparation,0
6,video01,train,150,6,Preparation,0
7,video01,train,175,7,Preparation,0
8,video01,train,200,8,Preparation,0
9,video01,train,225,9,Preparation,0


In [14]:
# ============================================================
# 14. Save 1 fps pilot annotation table
# ============================================================
pilot_ann_1fps_df.to_csv(SPLITS_DIR / "pilot_annotations_1fps.csv", index=False)
print("Saved:", SPLITS_DIR / "pilot_annotations_1fps.csv")

Saved: /Users/niranjani/Desktop/video-compression-project/cholec80/splits/pilot_annotations_1fps.csv


In [15]:
# ============================================================
# 15. Quick split summary
# ============================================================
split_summary_df = (
    pilot_ann_1fps_df.groupby(["split", "video"])
    .size()
    .reset_index(name="num_1fps_rows")
)

split_summary_df.head(20)

,split,video,num_1fps_rows
0,test,video16,2958
1,test,video17,1305
2,test,video18,1943
3,test,video19,2425
4,test,video20,1450
5,train,video01,1734
6,train,video02,2840
7,train,video03,5829
8,train,video04,1523
9,train,video05,2345


In [16]:
# ============================================================
# 16. Summary by split
# ============================================================
summary_by_split = (
    pilot_ann_1fps_df.groupby("split")
    .size()
    .reset_index(name="num_1fps_rows")
)

summary_by_split

,split,num_1fps_rows
0,test,10081
1,train,26956
2,val,9062


In [17]:
# ============================================================
# 17. Phase distribution by split
# ============================================================
phase_dist_by_split = (
    pilot_ann_1fps_df.groupby(["split", "phase_name"])
    .size()
    .reset_index(name="count")
    .sort_values(["split", "count"], ascending=[True, False])
)

phase_dist_by_split.head(30)

,split,phase_name,count
0,test,CalotTriangleDissection,4456
3,test,GallbladderDissection,2878
2,test,ClippingCutting,849
1,test,CleaningCoagulation,736
4,test,GallbladderPackaging,485
6,test,Preparation,344
5,test,GallbladderRetraction,333
7,train,CalotTriangleDissection,11640
10,train,GallbladderDissection,7636
8,train,CleaningCoagulation,1897


## Save split audit outputs

In [18]:
# ============================================================
# 18. Save split audit tables
# ============================================================
split_summary_df.to_csv(OUTPUTS_DIR / "split_audit" / "pilot_split_video_1fps_counts.csv", index=False)
summary_by_split.to_csv(OUTPUTS_DIR / "split_audit" / "pilot_split_totals_1fps.csv", index=False)
phase_dist_by_split.to_csv(OUTPUTS_DIR / "split_audit" / "pilot_split_phase_distribution_1fps.csv", index=False)

print("Saved split audit tables.")

Saved split audit tables.


## Final expected files

After this notebook, you should have:

### In `cholec80/splits/`
- `pilot_train_videos.csv`
- `pilot_val_videos.csv`
- `pilot_test_videos.csv`
- `pilot_split_videos.csv`
- `pilot_split.json`
- `pilot_annotations_fullfps.csv`
- `pilot_annotations_1fps.csv`

### In `cholec80/outputs/split_audit/`
- `pilot_split_video_1fps_counts.csv`
- `pilot_split_totals_1fps.csv`
- `pilot_split_phase_distribution_1fps.csv`